In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

In [2]:
latent_scaled = np.load(
    "../extract_latents/latent_scaled_len40_v2.npy"
)

print(latent_scaled.shape)

(7405, 64)


In [3]:
latent_scaled = torch.tensor(
    latent_scaled,
    dtype=torch.float32
)

print(latent_scaled.shape)

torch.Size([7405, 64])


In [4]:
dataset = TensorDataset(
    latent_scaled
)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True
)

print(len(loader))

29


In [5]:
T = 100
beta = torch.linspace(
    1e-4,
    0.02,
    T
)

alpha = 1 - beta

alpha_bar = torch.cumprod(
    alpha,
    dim=0
)

print(alpha_bar.shape)

torch.Size([100])


In [6]:
def q_sample(
    x0,
    t
):

    alpha_bar_t = alpha_bar.to(
        x0.device
    )

    noise = torch.randn_like(x0)

    sqrt_alpha_bar = (
        alpha_bar_t[t]
        .sqrt()
        .unsqueeze(1)
    )

    sqrt_one_minus_alpha_bar = (
        (1 - alpha_bar_t[t])
        .sqrt()
        .unsqueeze(1)
    )

    xt = (
        sqrt_alpha_bar * x0
        +
        sqrt_one_minus_alpha_bar * noise
    )

    return xt, noise

In [7]:
class DiffusionMLP(
    nn.Module
):

    def __init__(
        self,
        latent_dim=64
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                latent_dim + 1,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                latent_dim
            )
        )

    def forward(
        self,
        x,
        t
    ):

        t = (
            t.float()
            .unsqueeze(1)
            / T
        )

        x = torch.cat(
            [x,t],
            dim=1
        )

        return self.net(x)

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = DiffusionMLP().to(device)

print(device)

cuda


In [9]:
def diffusion_loss(
    pred_noise,
    true_noise
):

    return F.mse_loss(
        pred_noise,
        true_noise
    )

In [10]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [11]:
num_epochs = 100
for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for batch in loader:

        x0 = batch[0].to(device)

        batch_size = x0.shape[0]

        t = torch.randint(
            0,
            T,
            (batch_size,),
            device=device
        )

        xt, noise = q_sample(
            x0,
            t.cpu()
        )

        xt = xt.to(device)

        noise = noise.to(device)

        pred_noise = model(
            xt,
            t
        )

        loss = diffusion_loss(
            pred_noise,
            noise
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss={total_loss/len(loader):.6f}"
    )

Epoch [1/100] Loss=0.954837
Epoch [2/100] Loss=0.793944
Epoch [3/100] Loss=0.621026
Epoch [4/100] Loss=0.514297
Epoch [5/100] Loss=0.462237
Epoch [6/100] Loss=0.440471
Epoch [7/100] Loss=0.431534
Epoch [8/100] Loss=0.431744
Epoch [9/100] Loss=0.415848
Epoch [10/100] Loss=0.422812
Epoch [11/100] Loss=0.417269
Epoch [12/100] Loss=0.412836
Epoch [13/100] Loss=0.405326
Epoch [14/100] Loss=0.402262
Epoch [15/100] Loss=0.400494
Epoch [16/100] Loss=0.396821
Epoch [17/100] Loss=0.389366
Epoch [18/100] Loss=0.393381
Epoch [19/100] Loss=0.387029
Epoch [20/100] Loss=0.386218
Epoch [21/100] Loss=0.383935
Epoch [22/100] Loss=0.384025
Epoch [23/100] Loss=0.384687
Epoch [24/100] Loss=0.385487
Epoch [25/100] Loss=0.381767
Epoch [26/100] Loss=0.375833
Epoch [27/100] Loss=0.373638
Epoch [28/100] Loss=0.373707
Epoch [29/100] Loss=0.368714
Epoch [30/100] Loss=0.372002
Epoch [31/100] Loss=0.368232
Epoch [32/100] Loss=0.367794
Epoch [33/100] Loss=0.370186
Epoch [34/100] Loss=0.368414
Epoch [35/100] Loss=0.3

In [12]:
torch.save(
    model.state_dict(),
    "diffusion_len40_v2.pt"
)

In [13]:
@torch.no_grad()
def sample_latents(
    model,
    n_samples=10000
):

    model.eval()

    x = torch.randn(
        n_samples,
        64
    ).to(device)

    for t in reversed(
        range(T)
    ):

        t_tensor = torch.full(
            (n_samples,),
            t,
            device=device,
            dtype=torch.long
        )

        pred_noise = model(
            x,
            t_tensor
        )

        alpha_t = alpha[t].to(device)

        alpha_bar_t = (
            alpha_bar[t]
            .to(device)
        )

        beta_t = beta[t].to(device)

        if t > 0:

            noise = torch.randn_like(x)

        else:

            noise = torch.zeros_like(x)

        x = (
            (
                x
                -
                (
                    beta_t
                    /
                    torch.sqrt(
                        1-alpha_bar_t
                    )
                )
                *
                pred_noise
            )
            /
            torch.sqrt(alpha_t)
        )

        x += (
            torch.sqrt(beta_t)
            * noise
        )

    return x

In [14]:
generated_latents = sample_latents(
    model,
    n_samples=10000
)

print(
    generated_latents.shape
)

torch.Size([10000, 64])


In [15]:
generated_latents = (
    generated_latents
    .cpu()
    .numpy()
)

print(
    generated_latents.shape
)

(10000, 64)


In [16]:
import joblib

scaler = joblib.load(
    "../extract_latents/latent_scaler_len40_v2.pkl"
)

In [17]:
generated_latents = (
    scaler.inverse_transform(
        generated_latents
    )
)

In [18]:
print(generated_latents.shape)

print(generated_latents.mean())
print(generated_latents.std())

print(generated_latents.min())
print(generated_latents.max())

(10000, 64)
-0.073351115
0.4247217
-1.2667055
1.0554067


In [19]:
import numpy as np

np.save(
    "generated_latents_len40_v2.npy",
    generated_latents
)

print("Saved")

Saved
